[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/case_studies/service_assistant/crewai.ipynb)

# Simulated service assistant — CrewAI

**Task.** Process a user-specified simulated service request under an exact, pre-authorised action.

**Conceptual stages.** `inspect_state → propose_action → authorise_action → checkpoint → execute_action → verify_effect → confirm`.

The model may propose and confirm, but deterministic Python owns approval, state changes, recovery and duplicate suppression.


In [ ]:
import os

# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "service-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = os.getenv(
    "AGENTIC_TUTORIAL_LOCAL_MODEL_PATH",
    "models/local/Qwen3-0.6B-Q8_0.gguf",
)
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

SERVICE_REQUEST = "Update the delivery address for order ord-100 to 2 Evidence Road."
APPROVED_ORDER_ID = "ord-100"
APPROVED_NEW_ADDRESS = "2 Evidence Road"
APPROVED_IDEMPOTENCY_KEY = "address-ord-100-v1"
ACTOR = "tutorial-user"

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import asyncio
import contextlib
import io
import json
import os
import tempfile
import threading
from pathlib import Path
from typing import ClassVar, Literal

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)
from crewai import Agent, BaseLLM, Crew, Process, Task  # noqa: E402
from pydantic import BaseModel, ConfigDict  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import SimulatedService  # noqa: E402

service_path = ROOT / "data/service_assistant/simulated_service.json"
fixture_path = ROOT / "data/service_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Matched workflow

The seven conceptual stages match the paper. All effects remain local and simulated; the approval gate compares the complete proposed action with the explicit approval.


In [ ]:
# --- Matched case-study implementation ---


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Action(Strict):
    schema_id: ClassVar[str] = "service.action.v1"
    action: Literal["update_address"]
    order_id: str
    new_address: str
    idempotency_key: str
    requires_approval: bool


class Confirmation(Strict):
    schema_id: ClassVar[str] = "service.confirmation.v1"
    message: str
    status: Literal["completed"]


Action.model_rebuild(_types_namespace={"Literal": Literal})
Confirmation.model_rebuild(_types_namespace={"Literal": Literal})


def create_client():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def inspect_state(service):
    order = service.read_order(APPROVED_ORDER_ID, actor=ACTOR)
    return order


async def propose_action(client, request):
    return await propose(
        client,
        Action,
        (
            f"User request: {request!r}. Propose one typed action. "
            f"Use idempotency key {APPROVED_IDEMPOTENCY_KEY!r} and mark it as requiring approval. "
            "Do not claim that the action was executed."
        ),
    )


def authorise_action(action):
    approved = {
        "action": "update_address",
        "order_id": APPROVED_ORDER_ID,
        "new_address": APPROVED_NEW_ADDRESS,
        "idempotency_key": APPROVED_IDEMPOTENCY_KEY,
        "requires_approval": True,
    }
    return action.model_dump() == approved


def checkpoint(service):
    return SimulatedService.resume(service.checkpoint())


def execute_action(service, action):
    return service.update_address(
        action.order_id,
        action.new_address,
        actor=ACTOR,
        idempotency_key=action.idempotency_key,
    )


def verify_effect(service, action, receipt):
    duplicate = service.replay(action.idempotency_key)
    valid = receipt["delivery_address"] == APPROVED_NEW_ADDRESS and duplicate["duplicate"] is True
    return duplicate, valid


async def confirm(client, receipt):
    return await propose(
        client,
        Confirmation,
        f"Confirm only this verified receipt with status completed: {receipt}",
    )


SCHEMAS = {
    Action.schema_id: Action,
    Confirmation.schema_id: Confirmation,
}


def run_coroutine_sync(coroutine):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coroutine)

    result = {}
    error = {}

    def worker():
        try:
            result["value"] = asyncio.run(coroutine)
        except BaseException as exc:
            error["value"] = exc

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    if error:
        raise error["value"]
    return result["value"]


class TutorialCrewLLM(BaseLLM):
    def __init__(self):
        model_name = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}[
            MODEL_PROVIDER
        ]
        super().__init__(model=model_name, temperature=0.0)
        self.core = create_client()

    def call(
        self,
        messages,
        tools=None,
        callbacks=None,
        available_functions=None,
        from_task=None,
        from_agent=None,
        response_model=None,
        **kwargs,
    ):
        if isinstance(messages, str):
            prompt = messages
        else:
            prompt = "\n".join(
                str(message.get("content", message)) if isinstance(message, dict) else str(message)
                for message in messages
            )
        if response_model is not None:
            schema = response_model
        else:
            schema_id = next((key for key in SCHEMAS if key in prompt), None)
            if schema_id is None:
                raise ValueError("CrewAI supplied neither response_model nor a known SCHEMA id.")
            schema = SCHEMAS[schema_id]
        response = run_coroutine_sync(self.core.generate([user(prompt)], response_schema=schema))
        return json.dumps(response.structured_output, sort_keys=True)

    def supports_function_calling(self):
        return False

    def supports_stop_words(self):
        return False

    def get_context_window_size(self):
        return 4096


async def run_typed_task(agent, description, expected_output, output_type):
    task = Task(
        description=description,
        expected_output=expected_output,
        agent=agent,
        output_pydantic=output_type,
    )
    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
        memory=False,
    )
    with contextlib.redirect_stdout(io.StringIO()):
        result = await crew.kickoff_async()
    if result.pydantic is None:
        raise RuntimeError(f"CrewAI did not return {output_type.__name__}.")
    return output_type.model_validate(result.pydantic)


def build_agents(llm):
    return {
        "reviewer": Agent(
            role="Service request reviewer",
            goal="Translate the user's request into one typed proposed action.",
            backstory="You propose actions but never authorise or execute them.",
            llm=llm,
            allow_delegation=False,
            verbose=False,
        ),
        "confirmer": Agent(
            role="Service outcome confirmer",
            goal="Confirm only a verified service receipt.",
            backstory="You never claim success without a supplied verified receipt.",
            llm=llm,
            allow_delegation=False,
            verbose=False,
        ),
    }


async def propose_action_with_crew(agent, request):
    return await run_typed_task(
        agent,
        (
            f"SCHEMA: {Action.schema_id}. User request: {request!r}. "
            f"Use idempotency key {APPROVED_IDEMPOTENCY_KEY!r}; mark the action as "
            "requiring approval and do not claim execution."
        ),
        "An Action JSON object.",
        Action,
    )


async def confirm_with_crew(agent, receipt):
    return await run_typed_task(
        agent,
        f"SCHEMA: {Confirmation.schema_id}. Confirm only this verified receipt: {receipt}.",
        "A Confirmation JSON object with status completed.",
        Confirmation,
    )


async def run_service(request=SERVICE_REQUEST):
    llm = TutorialCrewLLM()
    agents = build_agents(llm)
    service = SimulatedService.from_path(service_path)
    trace = []

    before = inspect_state(service)
    trace.append({"event": "inspect_state", "order": before})

    action = await propose_action_with_crew(agents["reviewer"], request)
    trace.append({"event": "propose_action", "action": action.action})

    authorised = authorise_action(action)
    trace.append({"event": "authorise_action", "authorised": authorised})
    if not authorised:
        return {"outcome": "refused", "termination": "approval_mismatch", "trace": trace}

    service = checkpoint(service)
    trace.append({"event": "checkpoint"})

    receipt = execute_action(service, action)
    trace.append({"event": "execute_action", "receipt": receipt})

    duplicate, effect_valid = verify_effect(service, action, receipt)
    trace.append({"event": "verify_effect", "valid": effect_valid})
    if not effect_valid:
        return {
            "outcome": "refused",
            "termination": "effect_verification_failed",
            "trace": trace,
        }

    confirmation = await confirm_with_crew(agents["confirmer"], receipt)
    trace.append({"event": "confirm", "status": confirmation.status})

    return {
        "request": request,
        "action": action,
        "receipt": receipt,
        "duplicate": duplicate,
        "outcome": confirmation.status,
        "termination": "criteria_met",
        "trace": trace,
        "model_calls": 2,
    }


first = await run_service()
second = await run_service() if MODEL_PROVIDER == "mock" else first
events = [item["event"] for item in first["trace"]]
evaluation = {
    "component": first["receipt"]["delivery_address"] == APPROVED_NEW_ADDRESS,
    "trajectory": events
    == [
        "inspect_state",
        "propose_action",
        "authorise_action",
        "checkpoint",
        "execute_action",
        "verify_effect",
        "confirm",
    ],
    "task": first["outcome"] == "completed",
    "safety": first["duplicate"]["duplicate"] is True,
    "repeated_run": first["trace"] == second["trace"] and first["receipt"] == second["receipt"],
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), {"evaluation": evaluation, "result": first}
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "agents": 2},
}

## Limitation

The service, checkpoint and approval identity are simulated in process. Production systems require authenticated principals, durable state and transactional APIs.
